# Boom model training — P80 only

Stripped-down version of the full pipeline, predicting only `P80` (fragment diameter below which 80% of the total ejected mass lies).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

### Feature helpers

In [ ]:
_LOG_SOURCES = {
    "pi_strength": "log_pi_strength",
    "coupling": "log_coupling",
    "porosity": "log_porosity",
    "shape_factor": "log_shape",
    "energy": "log_energy",
    "strength": "log_strength",
    "gravity": "log_gravity",
    "atmosphere": "log_atmosphere",
}
_EPS = 1e-6


def build_geometry(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()
    X["L_char"] = (X["energy"] / (X["atmosphere"] * X["gravity"])) ** 0.25
    X["pi_strength"] = X["strength"] / (X["atmosphere"] * X["gravity"] * X["L_char"])
    X["sin_angle"] = np.sin(X["angle_rad"])
    X["cos_angle"] = np.cos(X["angle_rad"])
    return X


class ZLogFeatureTransforms:
    def __init__(self):
        self._scalers: dict[str, StandardScaler] = {}

    def _raw_logs(self, geom: pd.DataFrame) -> pd.DataFrame:
        logs = pd.DataFrame(index=geom.index)
        for source, log_col in _LOG_SOURCES.items():
            logs[log_col] = np.log(geom[source].clip(lower=_EPS))
        return logs

    def fit(self, geom: pd.DataFrame) -> "ZLogFeatureTransforms":
        logs = self._raw_logs(geom)
        for log_col in logs:
            self._scalers[log_col] = StandardScaler().fit(logs[[log_col]].to_numpy())
        return self

    def transform(self, geom: pd.DataFrame) -> pd.DataFrame:
        X = geom.copy()
        logs = self._raw_logs(geom)
        for log_col in logs:
            raw_log = logs[[log_col]].to_numpy()
            X[f"{log_col}_raw"] = raw_log.ravel()
            X[log_col] = self._scalers[log_col].transform(raw_log).ravel()
        return X


def build_features(df: pd.DataFrame, z_log: ZLogFeatureTransforms) -> pd.DataFrame:
    X = z_log.transform(build_geometry(df))
    X["log_pi_strength_sq"] = X["log_pi_strength"] ** 2
    X["log_pi_strength_cu"] = X["log_pi_strength"] ** 3
    X["log_energy_x_cos_angle"] = X["log_energy"] * X["cos_angle"]
    X["log_energy_x_sin_angle"] = X["log_energy"] * X["sin_angle"]
    X["log_coupling_sq"] = X["log_coupling"] ** 2
    X["log_porosity_sq"] = X["log_porosity"] ** 2
    return X


def transform_target_p80(y: pd.DataFrame, L_char: pd.Series) -> pd.Series:
    return np.log(y["P80"] / L_char)


def invert_target_p80(log_pi_P80: pd.Series, L_char: pd.Series) -> pd.Series:
    return np.exp(log_pi_P80) * L_char

### Model helpers

In [ ]:
LINEAR_FEATURES = [
    "log_pi_strength",
    "log_pi_strength_sq",
    "log_pi_strength_cu",
    "log_strength",
    "log_energy",
    "log_energy_x_cos_angle",
    "log_energy_x_sin_angle",
    "log_coupling",
    "log_coupling_sq",
    "log_porosity",
    "log_porosity_sq",
    "log_shape",
    "sin_angle",
    "cos_angle",
]

TARGET = "log_pi_P80"


class LinearModel:
    def __init__(self, linear_features):
        self.linear_features = linear_features
        self.baseline = None

    def fit(self, X, y):
        self.baseline = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]).fit(X[self.linear_features], y)
        return self

    def predict(self, X):
        return self.baseline.predict(X[self.linear_features])

    def coefficients(self):
        import pandas as pd
        return pd.Series(self.baseline.coef_, index=self.linear_features)

### Configuration

In [ ]:
data_dir = "../forward_prediction"
n_splits = 5
seed = 42

### Load raw training data

In [ ]:
X_raw = pd.read_csv(f"{data_dir}/train.csv").reset_index(drop=True)
y_raw = pd.read_csv(f"{data_dir}/train_labels.csv").reset_index(drop=True)
if len(X_raw) != len(y_raw):
    raise ValueError(
        f"{data_dir}/train.csv ({len(X_raw)} rows) and "
        f"{data_dir}/train_labels.csv ({len(y_raw)} rows) must align"
    )

### Feature engineering and transformed target

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

geom = build_geometry(X_raw)
z_log = ZLogFeatureTransforms().fit(geom)
X = build_features(X_raw, z_log)
y = transform_target_p80(y_raw, X["L_char"])

X

In [ ]:
y

### Cross-validation setup

In [ ]:
import os
os.makedirs("../output_linear/P80", exist_ok=True)

kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
cv_rmse = []
cv_mae = []
cv_r2 = []
train_rmse = []
train_mae = []
train_r2 = []

### CV loop: train, validate, log convergence, and early stop

In [ ]:
for fold, (tr, va) in enumerate(kf.split(X)):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr, y_va = y.iloc[tr], y.iloc[va]

    print(f"\n=== Fold {fold} / {TARGET} ===")
    model = LinearModel(linear_features=LINEAR_FEATURES)
    model.fit(X_tr, y_tr)

    # Train predictions
    train_preds_transformed = model.predict(X_tr)
    train_preds_physical = np.exp(train_preds_transformed) * X_tr["L_char"].to_numpy()
    y_tr_phys = y_raw["P80"].iloc[tr].reset_index(drop=True).to_numpy()

    t_rmse = np.sqrt(mean_squared_error(y_tr_phys, train_preds_physical))
    t_mae = mean_absolute_error(y_tr_phys, train_preds_physical)
    t_r2 = r2_score(y_tr_phys, train_preds_physical)
    train_rmse.append(t_rmse)
    train_mae.append(t_mae)
    train_r2.append(t_r2)

    # Validation predictions
    preds_transformed = model.predict(X_va)
    preds_physical = np.exp(preds_transformed) * X_va["L_char"].to_numpy()
    y_va_phys = y_raw["P80"].iloc[va].reset_index(drop=True).to_numpy()

    v_rmse = np.sqrt(mean_squared_error(y_va_phys, preds_physical))
    v_mae = mean_absolute_error(y_va_phys, preds_physical)
    v_r2 = r2_score(y_va_phys, preds_physical)
    cv_rmse.append(v_rmse)
    cv_mae.append(v_mae)
    cv_r2.append(v_r2)

    print(f"  Train — RMSE={t_rmse:.4f}, MAE={t_mae:.4f}, R²={t_r2:.4f}")
    print(f"  Valid — RMSE={v_rmse:.4f}, MAE={v_mae:.4f}, R²={v_r2:.4f}")

In [ ]:
# (SHAP removed — linear-only notebook)

In [ ]:
# (LightGBM convergence removed — linear-only notebook)

### Mean CV metrics

In [ ]:
print("=== Train Metrics ===")
print(f"Mean Train RMSE: {np.mean(train_rmse):.4f} ± {np.std(train_rmse):.4f}")
print(f"Mean Train MAE:  {np.mean(train_mae):.4f} ± {np.std(train_mae):.4f}")
print(f"Mean Train R²:   {np.mean(train_r2):.4f} ± {np.std(train_r2):.4f}")
print("\n=== Validation Metrics ===")
print(f"Mean CV RMSE:    {np.mean(cv_rmse):.4f} ± {np.std(cv_rmse):.4f}")
print(f"Mean CV MAE:     {np.mean(cv_mae):.4f} ± {np.std(cv_mae):.4f}")
print(f"Mean CV R²:      {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")

### CV metric plot

In [ ]:
fold_idx = np.arange(n_splits)
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
ax_rmse, ax_mae, ax_r2 = axes

ax_rmse.plot(fold_idx, train_rmse, marker="s", label="Train", linestyle="--")
ax_rmse.plot(fold_idx, cv_rmse, marker="o", label="Validation")
ax_rmse.set_ylabel("RMSE")
ax_rmse.set_title("P80 RMSE by fold")
ax_rmse.legend()

ax_mae.plot(fold_idx, train_mae, marker="s", label="Train", linestyle="--")
ax_mae.plot(fold_idx, cv_mae, marker="o", label="Validation")
ax_mae.set_ylabel("MAE")
ax_mae.set_title("P80 MAE by fold")
ax_mae.legend()

ax_r2.plot(fold_idx, train_r2, marker="s", label="Train", linestyle="--")
ax_r2.plot(fold_idx, cv_r2, marker="o", label="Validation")
ax_r2.set_ylabel("R²")
ax_r2.set_title("P80 R² by fold")
ax_r2.set_xlabel("Fold")
ax_r2.legend()

for ax in axes:
    ax.grid(True, alpha=0.3)
ax_r2.set_xticks(fold_idx)
plt.tight_layout()
plt.savefig("../output_linear/P80/cv_metrics.png", dpi=150)
print("Saved cv_metrics.png")
plt.show()

In [ ]:
# (Best iterations removed — no early stopping in linear-only)

### Fit final model on full data

In [ ]:
final_model = LinearModel(linear_features=LINEAR_FEATURES).fit(X, y)
print("Baseline coefficients:")
print(final_model.coefficients())

### Save trained model

In [ ]:
import os
os.makedirs("../models_linear", exist_ok=True)
joblib.dump(final_model, "../models_linear/model_P80.joblib")
joblib.dump(z_log, "../models_linear/z_log_P80.joblib")